# Session 6 | Capstone: 企业知识助手

**时长:** 3.5 小时 | **类型:** 80% 编码实战 + 20% 讨论

---

## 项目目标

整合三天所学,构建一个**完整的企业知识助手系统**:
- RAG 检索增强生成
- Agent 智能路由
- LLM-as-Judge 自动评测
- 多轮对话记忆

## 系统架构

```
用户提问
  → Agent(ReAct 循环)
    → 工具1: RAG 搜索(向量库 + 企业文档)
    → 工具2: 计算器
    → 工具3: 代码执行器
    → 工具4: 直接 LLM 回答
  → 带来源标注的回答
  → LLM-as-Judge 评测分数
```

## 6 个 Checkpoint

| # | Checkpoint | 目标 |
|---|-----------|------|
| 1 | 知识库检索 | 加载文档 → 嵌入 → 向量搜索 |
| 2 | RAG 回答 | 检索 + LLM 生成带来源的回答 |
| 3 | Agent 路由 | ReAct Agent 自动选择工具 |
| 4 | 评测打分 | LLM-as-Judge 自动评估质量 |
| 5 | 多轮对话 | 对话记忆 + 上下文引用 |
| 6 | 最终演示 | 自定义文档/工具 → 展示 |

## 完成后你将拥有

一个可以接入自己企业文档、回答专业问题、自动评估回答质量的 AI 助手原型。

## 场景设定

你是 **星辰科技有限公司** 的 AI 工程师,需要为公司内部构建一个知识助手,覆盖:

- **HR 制度**:年假、报销、考勤等
- **产品规格**:公司核心产品的技术参数
- **技术文档**:开发规范、API 说明

员工可以用自然语言提问,系统自动检索相关文档并生成回答。

---

## 环境准备

In [1]:
# 导入依赖
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), "utils"))
sys.path.insert(0, "utils")

import json
import re
import time
import numpy as np
from typing import List, Dict, Any, Optional, Callable
from dataclasses import dataclass, field

from embedding_backend import SimpleVectorStore
from config import setup

env = setup()

print("[OK] 基础模块导入完成")

[OK] 已加载配置: C:\Users\lvbab\Desktop\test2\enterprise_ai_training_3day\enterprise\.env
课程环境配置:
  API Key:   sk-...b2a9
  LLM:       dashscope / qwen-plus
  Embedding: dashscope / text-embedding-v3
[OK] 基础模块导入完成


In [2]:
# 初始化 LLM 和 Embedding（使用 Day0 配置）
llm = env.get_llm()
embedder = env.get_embedder()

print("[OK] LLM 和 Embedding 已初始化")

[LLM] dashscope / qwen-plus


[Embedding] dashscope / text-embedding-v3 (dim=1024)
[OK] LLM 和 Embedding 已初始化


---

# Step 1: 搭建知识库(0:15 - 0:30)

**目标:** 加载企业文档 → 分块 → 嵌入 → 存储到向量数据库

这是整个系统的数据基础。真实项目中文档可能来自 Confluence、飞书、PDF 等,这里我们用字符串模拟。

In [3]:
# 企业文档（模拟星辰科技有限公司内部资料）

COMPANY_DOCUMENTS = {
    "hr_leave": {
        "title": "年假与休假制度",
        "category": "HR",
        "content": """星辰科技年假制度(2024版)

一、年假天数
入职满1年不满10年:5天年假
入职满10年不满20年:10天年假
入职满20年以上:15天年假
试用期员工不享受年假。

二、请假流程
1. 通过 OA 系统提交年假申请
2. 直属主管审批(1-3天)
3. 部门总监审批(3天以上)
4. 年假可拆分使用,最小单位为半天

三、特殊规定
当年未休完的年假可结转至次年第一季度。
因工作需要无法休假的,按日薪300%发放补偿。
婚假15天,产假158天,陪产假15天。"""
    },
    "hr_expense": {
        "title": "费用报销制度",
        "category": "HR",
        "content": """星辰科技费用报销制度

一、差旅报销
交通:经济舱/高铁二等座(总监以上可商务舱)
住宿:一线城市不超过500元/晚,其他城市不超过350元/晚
餐饮:每日补贴100元,无需提供发票

二、办公用品
500元以下直接报销,500元以上需部门审批
电子设备需经IT部门审核

三、报销流程
1. OA系统提交报销单,附发票照片
2. 直属主管审批
3. 财务审核
4. 每月15日和30日统一打款

四、时限要求
费用发生后30天内提交报销,逾期不予受理。"""
    },
    "product_starlink": {
        "title": "StarLink 智能网关产品规格",
        "category": "产品",
        "content": """StarLink 智能网关 v3.0 产品规格

型号:SL-3000 / SL-3000 Pro

核心参数:
- CPU:ARM Cortex-A72 四核 1.8GHz
- 内存:SL-3000 为 2GB,SL-3000 Pro 为 4GB
- 存储:16GB eMMC + SD卡扩展
- 网络:千兆以太网 x4,WiFi 6,5G 模块(Pro版)
- 协议:MQTT, CoAP, HTTP/2, Modbus TCP
- 边缘计算:支持 TensorFlow Lite 推理,最大支持 50 TOPS

使用场景:
- 工厂设备数据采集与边缘计算
- 智慧园区网关
- 车联网数据汇聚

价格:SL-3000 2999元 / SL-3000 Pro 4999元
质保期:2年"""
    },
    "product_starview": {
        "title": "StarView 数据可视化平台",
        "category": "产品",
        "content": """StarView 数据可视化平台 v2.5

功能概述:
- 实时数据大屏,支持100+数据源接入
- 拖拽式仪表盘配置,无需编码
- 内置30+图表组件(折线图、柱状图、热力图、地理图等)
- 支持自定义 SQL 查询和 Python 脚本

技术架构:
- 前端:React + D3.js + WebSocket
- 后端:Go + ClickHouse + Redis
- 部署:支持私有化部署和 SaaS 版本

授权方式:
- 基础版:免费(5个大屏以内,10个数据源以内)
- 企业版:50000元/年(不限大屏和数据源)
- 旗舰版:150000元/年(含技术支持和定制开发)"""
    },
    "tech_api": {
        "title": "内部 API 开发规范",
        "category": "技术",
        "content": """星辰科技 API 开发规范 v1.2

一、RESTful 规范
- 使用 HTTPS,禁止 HTTP
- URL 使用小写字母和连字符:/api/v1/user-profiles
- GET 查询、POST 创建、PUT 更新、DELETE 删除
- 状态码:200成功、201创建、400参数错误、401未授权、404不存在、500服务器错误

二、认证方式
- 内部服务:JWT Token,有效期 2 小时
- 外部 API:OAuth 2.0 + API Key
- 所有 Token 必须通过 Header 传递:Authorization: Bearer <token>

三、性能要求
- P99 延迟不超过 200ms
- QPS 不低于 1000(单实例)
- 必须实现限流和熔断

四、文档要求
- 使用 Swagger/OpenAPI 3.0 生成文档
- 每个接口必须有请求/响应示例"""
    },
    "tech_deploy": {
        "title": "部署与运维规范",
        "category": "技术",
        "content": """星辰科技部署与运维规范

一、容器化要求
- 所有服务必须 Docker 容器化
- 生产环境使用 Kubernetes 编排
- 镜像大小限制:基础服务不超过200MB,AI 服务不超过2GB

二、CI/CD 流程
- 代码提交 -> 自动化测试 -> 代码审查 -> 构建镜像 -> 灰度发布 -> 全量上线
- 测试覆盖率不低于80%
- 灰度比例:5% -> 20% -> 50% -> 100%

三、监控告警
- 基础设施:Prometheus + Grafana
- 日志:ELK(Elasticsearch + Logstash + Kibana)
- 告警:PagerDuty 集成,P0 事件 15 分钟内响应

四、数据库规范
- MySQL:单表不超过2000万行,超过则分库分表
- Redis:缓存 TTL 不超过 24 小时
- 所有数据库变更需经 DBA 审核"""
    }
}

print(f"[OK] 已准备 {len(COMPANY_DOCUMENTS)} 份企业文档")
for doc_id, doc in COMPANY_DOCUMENTS.items():
    print(f"  - [{doc['category']}] {doc['title']}({len(doc['content'])}字)")

[OK] 已准备 6 份企业文档
  - [HR] 年假与休假制度(234字)
  - [HR] 费用报销制度(227字)
  - [产品] StarLink 智能网关产品规格(345字)
  - [产品] StarView 数据可视化平台(296字)
  - [技术] 内部 API 开发规范(405字)
  - [技术] 部署与运维规范(397字)


In [4]:
# 文档分块器（Document Chunker）
# LLM 上下文窗口有限，整篇文档塞不进去。
# 分块后检索时只取最相关的片段，节省 token 又提高精准度。

class DocumentChunker:
    """将长文档切分为适合检索的小块"""

    def __init__(self, chunk_size: int = 300, chunk_overlap: int = 50):
        self.chunk_size = chunk_size        # 每块字符数
        self.chunk_overlap = chunk_overlap  # 相邻块重叠字符数（防止信息截断）

    def chunk_text(self, text: str, metadata: Dict = None) -> List[Dict]:
        """按字符长度切块，带重叠"""
        if metadata is None:
            metadata = {}
        chunks = []
        start = 0
        while start < len(text):
            end = start + self.chunk_size
            chunk_text = text[start:end]
            chunks.append({
                "text": chunk_text,
                "metadata": {
                    **metadata,
                    "chunk_index": len(chunks),
                    "start_char": start,
                    "end_char": min(end, len(text))
                }
            })
            start += self.chunk_size - self.chunk_overlap
        return chunks


# 分块所有文档
chunker = DocumentChunker(chunk_size=300, chunk_overlap=50)

all_chunks = []
all_metadata = []

for doc_id, doc in COMPANY_DOCUMENTS.items():
    chunks = chunker.chunk_text(
        doc["content"],
        metadata={"doc_id": doc_id, "title": doc["title"], "category": doc["category"]}
    )
    for chunk in chunks:
        all_chunks.append(chunk["text"])
        all_metadata.append(chunk["metadata"])

print(f"[OK] 共 {len(COMPANY_DOCUMENTS)} 份文档 -> {len(all_chunks)} 个块")

[OK] 共 6 份文档 -> 10 个块


In [5]:
# 构建向量数据库
# SimpleVectorStore 内部流程:
#   文本 → Embedding 向量 → 存储 (向量, 文本, 元信息)
#   搜索时: query → 向量 → 余弦相似度 → Top-K

vector_store = SimpleVectorStore(embedder)
vector_store.add_documents(all_chunks, metadata=all_metadata)

# 快速验证
results = vector_store.search("年假有几天", top_k=2)
print("\n[验证] 搜索 '年假有几天':")
if results and len(results) > 0:
    for r in results:
        print(f"  [{r['score']:.3f}] {r['document'][:80]}...")
    print("✅ 向量数据库构建正常")
else:
    print("❌ 搜索未返回结果，请检查向量数据库构建步骤")

✓ 已添加 10 个文档，总计 10 个

[验证] 搜索 '年假有几天':
  [0.706] 星辰科技年假制度(2024版)

一、年假天数
入职满1年不满10年:5天年假
入职满10年不满20年:10天年假
入职满20年以上:15天年假
试用期员工不享...
  [0.460] 星辰科技费用报销制度

一、差旅报销
交通:经济舱/高铁二等座(总监以上可商务舱)
住宿:一线城市不超过500元/晚,其他城市不超过350元/晚
餐饮:每日补贴...
✅ 向量数据库构建正常


---

## Checkpoint 1: 知识库检索

**目标:** 实现一个 `search_knowledge` 函数,接收查询字符串,返回格式化的搜索结果。

### 思路提示

回顾上面 `vector_store.search()` 的用法——它返回一个列表,每项包含 `document`、`score`、`metadata`。
你需要在此基础上加一层**相关性过滤**:低于阈值的结果应该被丢弃,避免返回不相关的内容。

要求:
1. 调用 `vector_store.search()` 获取候选结果
2. 设置相关性阈值（threshold），过滤低分结果
3. 返回包含文档内容、分数和元数据的结果列表
4. 处理无相关文档的情况（返回空列表）

**预期输出**：
```
搜索 "请假流程": 
  [1] (0.xx) 相关文档片段...
  [2] (0.xx) 相关文档片段...
过滤后（阈值>0.3）: X条结果
```

In [6]:
# Checkpoint 1: 实现知识库检索函数
# 提示: vector_store.search(query, top_k=k, threshold=t) 返回
#   [{"document": str, "score": float, "metadata": dict}, ...]

def search_knowledge(query: str, top_k: int = 3, threshold: float = 0.3) -> List[Dict]:
    """
    搜索企业知识库

    Args:
        query: 用户查询
        top_k: 返回条数
        threshold: 最低分阈值
    Returns:
        List[Dict]: 搜索结果，每项包含 document, score, metadata
    """
    # ↓↓↓ 你的代码 ↓↓↓
    results = vector_store.search(query, top_k=top_k, threshold=threshold)
    return [r for r in results if r["score"] >= threshold]
    # ↑↑↑ 你的代码 ↑↑↑

# === 测试 ===
test_queries = [
    "年假有几天？",
    "StarLink 基础版的价格是多少？",
    "API 认证用什么方式？",
    "今天天气怎么样？",
]

for q in test_queries:
    results = search_knowledge(q)
    print(f"\n[查询] {q}")
    if results:
        for r in results:
            print(f"  [{r['score']:.3f}] [{r['metadata'].get('category', '?')}] {r['document'][:60]}...")
    else:
        print("  -> 未找到相关文档")

# 验证
related_results = search_knowledge("年假有几天？")
unrelated_results = search_knowledge("今天天气怎么样？")

if related_results and len(related_results) > 0:
    print("\n✅ 检查点通过：相关查询能返回结果")
else:
    print("\n❌ 检查点失败：相关查询未返回结果")

if len(unrelated_results) == 0:
    print("✅ 检查点通过：无关查询正确返回空列表")
else:
    print("⚠️ 提示：无关查询仍返回了结果，可考虑提高 threshold")


[查询] 年假有几天？
  [0.710] [HR] 星辰科技年假制度(2024版)

一、年假天数
入职满1年不满10年:5天年假
入职满10年不满20年:10天年假
入职...
  [0.457] [HR] 星辰科技费用报销制度

一、差旅报销
交通:经济舱/高铁二等座(总监以上可商务舱)
住宿:一线城市不超过500元/晚,其...
  [0.442] [产品] 50000元/年(不限大屏和数据源)
- 旗舰版:150000元/年(含技术支持和定制开发)...

[查询] StarLink 基础版的价格是多少？
  [0.681] [产品] StarLink 智能网关 v3.0 产品规格

型号:SL-3000 / SL-3000 Pro

核心参数:
- C...
  [0.652] [产品] StarView 数据可视化平台 v2.5

功能概述:
- 实时数据大屏,支持100+数据源接入
- 拖拽式仪表盘配置...
  [0.624] [产品] 50000元/年(不限大屏和数据源)
- 旗舰版:150000元/年(含技术支持和定制开发)...



[查询] API 认证用什么方式？
  [0.670] [技术] 星辰科技 API 开发规范 v1.2

一、RESTful 规范
- 使用 HTTPS,禁止 HTTP
- URL 使用...
  [0.584] [技术] 通过 Header 传递:Authorization: Bearer <token>

三、性能要求
- P99 延迟不...
  [0.501] [技术] 星辰科技部署与运维规范

一、容器化要求
- 所有服务必须 Docker 容器化
- 生产环境使用 Kubernetes...

[查询] 今天天气怎么样？
  [0.396] [产品] StarView 数据可视化平台 v2.5

功能概述:
- 实时数据大屏,支持100+数据源接入
- 拖拽式仪表盘配置...
  [0.386] [技术] 星辰科技 API 开发规范 v1.2

一、RESTful 规范
- 使用 HTTPS,禁止 HTTP
- URL 使用...
  [0.377] [产品]  50 TOPS

使用场景:
- 工厂设备数据采集与边缘计算
- 智慧园区网关
- 车联网数据汇聚

价格:SL-30...



✅ 检查点通过：相关查询能返回结果
⚠️ 提示：无关查询仍返回了结果，可考虑提高 threshold


---

# Step 2: 构建 RAG 回答系统(0:30 - 0:45)

**目标:** 检索相关文档 → 注入上下文 → LLM 生成带来源标注的回答

在 Step 1 中,我们实现了向量检索——给定一个问题,能找到最相关的文档片段。但光有片段还不够,我们需要把这些片段**拼接成上下文**,交给 LLM 来生成一段完整的、可读的回答。这就是 **RAG（Retrieval-Augmented Generation）** 的核心流程:

```
用户问题 → 向量搜索 → Top-K 文档片段 → 拼接为 context → LLM 生成回答
```

下面的 `KnowledgeRAG` 类将检索和生成封装在一起。请仔细阅读代码中的注释,理解每个方法的作用。

In [7]:
# RAG 系统：检索 + 生成

class KnowledgeRAG:
    """企业知识库 RAG 系统"""

    SYSTEM_PROMPT = (
        "你是星辰科技的内部知识助手。请根据提供的参考资料回答员工的问题。\n\n"
        "规则:\n"
        "1. 只使用参考资料中的信息回答,不要编造\n"
        "2. 如果参考资料中没有相关信息,明确告知用户\n"
        "3. 回答要简洁准确\n"
        "4. 在回答末尾标注信息来源\n\n"
        "参考资料:\n{context}"
    )

    def __init__(self, vector_store, llm):
        self.vector_store = vector_store
        self.llm = llm

    def retrieve(self, query: str, top_k: int = 3, threshold: float = 0.3) -> List[Dict]:
        """检索相关文档"""
        results = self.vector_store.search(query, top_k=top_k, threshold=threshold)
        return [r for r in results if r["score"] >= threshold]

    def format_context(self, results: List[Dict]) -> str:
        """将检索结果格式化为上下文字符串"""
        if not results:
            return "(未找到相关参考资料)"
        parts = []
        for i, r in enumerate(results, 1):
            title = r["metadata"].get("title", "未知")
            parts.append(f"[来源{i}: {title}]\n{r['document']}")
        return "\n\n".join(parts)

    def answer(self, query: str, top_k: int = 3) -> Dict:
        """RAG 完整流程：检索 → 拼接上下文 → LLM 生成"""
        results = self.retrieve(query, top_k=top_k)
        context = self.format_context(results)
        system_msg = self.SYSTEM_PROMPT.format(context=context)
        messages = [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": query}
        ]
        response = self.llm.chat(messages, temperature=0.3)
        return {
            "query": query,
            "answer": response,
            "sources": [{"title": r["metadata"].get("title"), "score": r["score"]} for r in results],
            "context_used": context
        }


rag = KnowledgeRAG(vector_store, llm)
print("[OK] RAG 系统已初始化")

[OK] RAG 系统已初始化


---

## Checkpoint 2: RAG 回答

**目标:** 测试 RAG 系统,观察检索质量和回答质量。

### 思路提示

好的测试应该覆盖**正面情况**和**边界情况**:
- 正面情况：问题在知识库覆盖范围内,应该能准确回答
- 边界情况：问题超出知识库范围,系统应该诚实地说"不知道"而不是编造

要求:
1. 用至少 3 个问题测试 RAG 系统
2. 至少包含一个「无法回答」的问题（超出知识库范围）
3. 打印回答和来源，观察检索是否命中了正确的文档

**预期输出**：
```
Q: 公司的年假制度是什么？
A: 根据公司规定...（包含具体信息的回答）

Q: [越界问题]
A: 根据现有知识库，未找到相关信息...
```

**要求**：至少添加2个你自己设计的测试问题，测试RAG系统的边界。

In [8]:
# Checkpoint 2: 测试 RAG 系统

# ↓↓↓ 你的代码 ↓↓↓
test_questions = [
    "入职5年有几天年假?",
    "出差住宿标准是多少?",
    "StarLink 基础版支持哪些协议?",
    "StarView 企业版多少钱?",
    "公司有没有加密货币产品有哪些?",
    "报销流程是什么?",
    "部署服务需要容器化吗?",
]
# ↑↑↑ 你的代码 ↑↑↑

cp2_has_answer = 0
cp2_has_source = 0

for q in test_questions:
    result = rag.answer(q)
    print(f"\n{'='*60}")
    print(f"[问题] {result['query']}")
    print(f"[回答] {result['answer']}")
    if result['sources']:
        print(f"[来源] {', '.join(s['title'] for s in result['sources'])}")
        cp2_has_source += 1
    else:
        print("[来源] 无（未检索到相关文档）")
    if result['answer'] and len(result['answer'].strip()) > 0:
        cp2_has_answer += 1

if cp2_has_answer >= 4:
    print(f"\n✅ 检查点通过：RAG 系统正常，{cp2_has_answer}/{len(test_questions)} 个问题获得了回答")
else:
    print(f"\n❌ 检查点失败：仅 {cp2_has_answer}/{len(test_questions)} 个问题获得回答")

if cp2_has_source >= 4:
    print(f"✅ 检查点通过：{cp2_has_source}/{len(test_questions)} 个问题检索到了来源文档")
else:
    print(f"❌ 检查点失败：仅 {cp2_has_source}/{len(test_questions)} 个问题有来源")


[问题] 入职5年有几天年假?
[回答] 入职5年有5天年假。  
（依据：[来源1: 年假与休假制度]中“一、年假天数”规定：入职满1年不满10年，享有5天年假）
[来源] 年假与休假制度, StarView 数据可视化平台, 费用报销制度



[问题] 出差住宿标准是多少?
[回答] 出差住宿标准为：一线城市不超过500元/晚，其他城市不超过350元/晚。  
[来源1: 费用报销制度]
[来源] 费用报销制度, StarView 数据可视化平台, 年假与休假制度



[问题] StarLink 基础版支持哪些协议?
[回答] StarLink 智能网关（包括 SL-3000 和 SL-3000 Pro）支持的协议为：MQTT、CoAP、HTTP/2、Modbus TCP。  
注：资料中未提及“基础版”这一型号或版本划分，所有协议支持均基于 StarLink 智能网关 v3.0 产品规格（含 SL-3000 和 SL-3000 Pro），两者协议支持一致。  
[来源1: StarLink 智能网关产品规格]
[来源] StarLink 智能网关产品规格, StarView 数据可视化平台, 内部 API 开发规范



[问题] StarView 企业版多少钱?
[回答] StarView 企业版价格为50000元/年（不限大屏和数据源）。  
[来源1]
[来源] StarView 数据可视化平台, StarView 数据可视化平台, StarLink 智能网关产品规格



[问题] 公司有没有加密货币产品有哪些?
[回答] 参考资料中未提及星辰科技有任何加密货币相关产品。
[来源] StarLink 智能网关产品规格, StarView 数据可视化平台, StarView 数据可视化平台



[问题] 报销流程是什么?
[回答] 报销流程如下：  
1. 在OA系统提交报销单，并附上发票照片；  
2. 由直属主管审批；  
3. 财务部门审核；  
4. 每月15日和30日统一打款。  

此外，费用发生后须在30天内提交报销，逾期不予受理。  

信息来源：[来源1: 费用报销制度]
[来源] 费用报销制度, StarView 数据可视化平台, 年假与休假制度



[问题] 部署服务需要容器化吗?
[回答] 是的，所有服务必须 Docker 容器化。  
信息来源：[来源1: 部署与运维规范]
[来源] 部署与运维规范, 内部 API 开发规范, 部署与运维规范

✅ 检查点通过：RAG 系统正常，7/7 个问题获得了回答
✅ 检查点通过：7/7 个问题检索到了来源文档


---

# Step 3: 添加 Agent 层(0:45 - 1:00)

**目标:** 构建 ReAct Agent,让它自动判断用什么工具回答问题

### 为什么需要 Agent?

Step 2 的 RAG 系统只能做一件事——从文档里检索信息再回答。但员工的问题是多种多样的:

| 问题类型 | 示例 | 需要的工具 |
|----------|------|-----------|
| 企业知识 | "报销流程是什么?" | RAG 检索 |
| 数学计算 | "5天住宿费总共多少?" | 计算器 |
| 代码编写 | "写段Python读CSV" | 代码执行器 |
| 通用问题 | "今天星期几?" | 直接LLM回答 |

Agent 的作用就是做**路由**——分析问题类型,自动选择合适的工具。这里我们使用 **ReAct（Reasoning + Acting）** 范式:先推理(Thought),再行动(Action),观察结果(Observation),循环直到得出最终答案。

In [9]:
# 定义工具（Agent 可调用的能力）

@dataclass
class Tool:
    """工具定义：名称 + 描述 + 参数规格 + 执行函数"""
    name: str
    description: str
    parameters: Dict[str, Any]
    func: Callable

    def run(self, **kwargs) -> str:
        try:
            result = self.func(**kwargs)
            return str(result)
        except Exception as e:
            return f"Error: {type(e).__name__}: {str(e)}"


# 工具1: RAG 搜索
def rag_search(query: str) -> str:
    """搜索企业知识库并返回相关内容"""
    result = rag.answer(query)
    sources = ", ".join(s["title"] for s in result["sources"])
    return f"{result['answer']}\n\n[来源: {sources}]"


# 工具2: 计算器（安全的数学表达式求值）
def calculator(expression: str) -> str:
    """安全的数学计算器"""
    import math
    safe_dict = {
        'sqrt': math.sqrt, 'sin': math.sin, 'cos': math.cos,
        'log': math.log, 'pi': math.pi, 'e': math.e,
        'abs': abs, 'round': round, 'pow': pow, 'min': min, 'max': max,
    }
    allowed = set('0123456789+-*/().eE ,')
    clean = expression.replace(' ', '')
    for char in clean:
        if char not in allowed and not char.isalpha():
            return f"不允许的字符: {char}"
    try:
        result = eval(expression, {"__builtins__": {}}, safe_dict)
        return f"{expression} = {result}"
    except Exception as e:
        return f"计算错误: {e}"


# 工具3: 代码执行器（受限沙箱）
def code_executor(code: str) -> str:
    """执行 Python 代码片段（安全沙箱）"""
    import io, contextlib
    forbidden = ['import os', 'import sys', 'open(', '__import__',
                 'exec(', 'eval(', 'subprocess', 'shutil']
    for f in forbidden:
        if f in code:
            return f"安全限制：不允许使用 {f}"
    stdout = io.StringIO()
    try:
        with contextlib.redirect_stdout(stdout):
            exec(code, {"__builtins__": {"print": print, "range": range,
                                          "len": len, "str": str, "int": int,
                                          "float": float, "list": list, "dict": dict,
                                          "sum": sum, "min": min, "max": max,
                                          "sorted": sorted, "enumerate": enumerate}})
        output = stdout.getvalue()
        return output if output else "(代码执行完毕,无输出)"
    except Exception as e:
        return f"执行错误: {type(e).__name__}: {e}"


# 工具4: 直接 LLM 回答
def direct_answer(question: str) -> str:
    """通用问题直接用 LLM 回答（不检索知识库）"""
    resp = llm.chat([{"role": "user", "content": question}], temperature=0.5)
    return resp


# 注册工具
tools = [
    Tool("rag_search", "搜索企业知识库(HR制度、产品信息、技术文档)",
         {"type": "object", "properties": {"query": {"type": "string", "description": "搜索查询"}}},
         rag_search),
    Tool("calculator", "数学计算(加减乘除、函数等)",
         {"type": "object", "properties": {"expression": {"type": "string", "description": "数学表达式"}}},
         calculator),
    Tool("code_executor", "执行 Python 代码片段",
         {"type": "object", "properties": {"code": {"type": "string", "description": "Python 代码"}}},
         code_executor),
    Tool("direct_answer", "直接用 AI 回答通用问题(不需要检索时使用)",
         {"type": "object", "properties": {"question": {"type": "string", "description": "问题"}}},
         direct_answer),
]

print("[OK] 已注册工具:")
for t in tools:
    print(f"  - {t.name}: {t.description}")

[OK] 已注册工具:
  - rag_search: 搜索企业知识库(HR制度、产品信息、技术文档)
  - calculator: 数学计算(加减乘除、函数等)
  - code_executor: 执行 Python 代码片段
  - direct_answer: 直接用 AI 回答通用问题(不需要检索时使用)


In [10]:
# ReAct Agent：推理 + 行动循环
# 核心思想: Thought → Action → Observation → ... → Final Answer

class KnowledgeAgent:
    """企业知识助手 Agent（ReAct 范式）"""

    SYSTEM_PROMPT = (
        "你是星辰科技的智能助手,可以使用工具来回答问题。\n\n"
        "可用工具:\n{tool_descriptions}\n\n"
        "使用以下格式:\n"
        "Thought: [分析问题,决定使用哪个工具]\n"
        "Action: [工具名称]\n"
        "Action Input: {{\"key\": \"value\"}}\n\n"
        "观察到工具返回结果后继续思考:\n"
        "Thought: [根据结果进行分析]\n"
        "Final Answer: [最终回答]\n\n"
        "重要规则:\n"
        "- 企业相关问题优先使用 rag_search\n"
        "- 需要数学计算时使用 calculator\n"
        "- Action Input 必须是合法 JSON\n"
        "- 最终必须给出 Final Answer"
    )

    def __init__(self, llm, tools: List[Tool], max_steps: int = 5):
        self.llm = llm
        self.tools = {t.name: t for t in tools}
        self.max_steps = max_steps

    def _get_tool_descriptions(self) -> str:
        """生成工具描述文本"""
        descs = []
        for name, tool in self.tools.items():
            params = tool.parameters.get("properties", {})
            param_str = ", ".join(f"{k}: {v.get('description', '')}" for k, v in params.items())
            descs.append(f"- {name}: {tool.description}  (参数: {param_str})")
        return "\n".join(descs)

    def _parse_response(self, response: str) -> Dict:
        """解析 LLM 输出，提取 Thought / Action / Final Answer"""
        result = {"thought": None, "action": None, "action_input": None, "final_answer": None}

        thought_m = re.search(r'Thought:\s*(.+?)(?=Action:|Final Answer:|$)', response, re.DOTALL)
        if thought_m:
            result["thought"] = thought_m.group(1).strip()

        final_m = re.search(r'Final Answer:\s*(.+?)$', response, re.DOTALL)
        if final_m:
            result["final_answer"] = final_m.group(1).strip()
            return result

        action_m = re.search(r'Action:\s*([\w_]+)', response)
        if action_m:
            result["action"] = action_m.group(1).strip()

        input_m = re.search(r'Action Input:\s*(.+?)(?=Thought:|Action:|Final Answer:|$)', response, re.DOTALL)
        if input_m:
            input_str = input_m.group(1).strip()
            try:
                result["action_input"] = json.loads(input_str)
            except json.JSONDecodeError:
                result["action_input"] = {"input": input_str}

        return result

    def run(self, question: str, verbose: bool = True) -> Dict:
        """运行 Agent（ReAct 循环）"""
        tool_descs = self._get_tool_descriptions()
        system_msg = self.SYSTEM_PROMPT.format(tool_descriptions=tool_descs)

        messages = [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": question}
        ]

        steps = []
        for step in range(self.max_steps):
            response = self.llm.chat(messages, temperature=0.1)
            parsed = self._parse_response(response)

            if verbose:
                print(f"\n--- Step {step+1} ---")
                if parsed["thought"]:
                    print(f"Thought: {parsed['thought']}")

            if parsed["final_answer"]:
                if verbose:
                    print(f"Final Answer: {parsed['final_answer']}")
                return {
                    "question": question,
                    "answer": parsed["final_answer"],
                    "steps": steps,
                    "num_steps": step + 1
                }

            if parsed["action"] and parsed["action"] in self.tools:
                tool = self.tools[parsed["action"]]
                action_input = parsed["action_input"] or {}
                if verbose:
                    print(f"Action: {parsed['action']}({action_input})")

                observation = tool.run(**action_input)
                if verbose:
                    obs_display = observation[:200] + "..." if len(observation) > 200 else observation
                    print(f"Observation: {obs_display}")

                steps.append({"action": parsed["action"], "input": action_input, "observation": observation})
                messages.append({"role": "assistant", "content": response})
                messages.append({"role": "user", "content": f"Observation: {observation}"})
            else:
                return {
                    "question": question,
                    "answer": response,
                    "steps": steps,
                    "num_steps": step + 1
                }

        return {"question": question, "answer": "达到最大步骤数,无法完成回答", "steps": steps, "num_steps": self.max_steps}


agent = KnowledgeAgent(llm, tools)
print("[OK] 企业知识 Agent 已初始化")

[OK] 企业知识 Agent 已初始化


---

## Checkpoint 3: Agent 路由

**目标:** 测试 Agent 能否正确选择工具。

### 思路提示

准备 4 类问题,每类至少 1 个。关注 Agent 输出中的 `Action` 字段——它告诉你 Agent 选择了哪个工具:
1. 企业知识问题 → 应使用 `rag_search`
2. 数学计算问题 → 应使用 `calculator`
3. 代码编写问题 → 应使用 `code_executor`
4. 通用问题 → 应使用 `direct_answer`

如果 Agent 选错了工具,思考一下:是问题表述不够清晰,还是工具描述需要优化?

**预期输出**：
```
知识问题 → 使用 rag_search 工具
计算问题 → 使用 calculator 工具
代码问题 → 使用 code_executor 工具
直接问答 → 使用 direct_answer 工具
```

In [11]:
# Checkpoint 3: 测试 Agent 路由

# ↓↓↓ 你的代码 ↓↓↓
agent_test_cases = [
    "公司的年假制度是怎样的?",                                     # → rag_search
    "出差5天,住宿每天500元,交通补贴每天100元,总共多少?",             # → calculator
    "请帮我写一个Python函数,计算1到100的偶数列表并打印前10个",        # → code_executor
    "Python 的 list comprehension 怎么用?",                        # → direct_answer
]

expected_tools = ["rag_search", "calculator", "code_executor", "direct_answer"]
# ↑↑↑ 你的代码 ↑↑↑

cp3_results = []
for q, expected in zip(agent_test_cases, expected_tools):
    print(f"\n{'='*60}")
    print(f"[问题] {q}")
    print(f"[预期工具] {expected}")
    result = agent.run(q, verbose=True)
    print(f"\n[最终回答] {result['answer'][:200]}")
    print(f"[使用步数] {result['num_steps']}")
    cp3_results.append(result)

# 验证
cp3_answered = sum(1 for r in cp3_results if r.get('answer') and len(r['answer'].strip()) > 0)
cp3_reasonable_steps = sum(1 for r in cp3_results if 0 < r.get('num_steps', 0) <= 5)

if cp3_answered == len(agent_test_cases):
    print(f"\n✅ 检查点通过：Agent 成功回答了全部 {cp3_answered}/{len(agent_test_cases)} 个问题")
else:
    print(f"\n❌ 检查点失败：Agent 仅回答了 {cp3_answered}/{len(agent_test_cases)} 个问题")

if cp3_reasonable_steps == len(agent_test_cases):
    print(f"✅ 检查点通过：所有问题在合理步数内完成")
else:
    print(f"⚠️ 提示：{len(agent_test_cases) - cp3_reasonable_steps} 个问题步数异常")


[问题] 公司的年假制度是怎样的?
[预期工具] rag_search



--- Step 1 ---
Thought: 这是一个关于公司HR制度的问题，需要查询企业知识库中的HR制度相关内容。
Final Answer: 星辰科技的年假制度如下：  
- 员工入职满1年不满10年，享有5天带薪年假；  
- 入职满10年不满20年，享有10天带薪年假；  
- 入职满20年及以上，享有15天带薪年假。  
年假可分段使用，但需提前3个工作日通过OA系统申请并获直属上级审批；未休完年假原则上不跨年度累计（特殊情况经HRBP审批后可延期至次年3月31日前）。此外，国家法定节假日、周末及因公出差时间不计入年假天数。

[最终回答] 星辰科技的年假制度如下：  
- 员工入职满1年不满10年，享有5天带薪年假；  
- 入职满10年不满20年，享有10天带薪年假；  
- 入职满20年及以上，享有15天带薪年假。  
年假可分段使用，但需提前3个工作日通过OA系统申请并获直属上级审批；未休完年假原则上不跨年度累计（特殊情况经HRBP审批后可延期至次年3月31日前）。此外，国家法定节假日、周末及因公出差时间不计入年假天数。
[使用步数] 1

[问题] 出差5天,住宿每天500元,交通补贴每天100元,总共多少?
[预期工具] calculator



--- Step 1 ---
Thought: 这是一个简单的数学计算问题，需要计算5天的住宿费和交通补贴总和。住宿每天500元，交通补贴每天100元，总共5天，所以总费用为：(500 + 100) × 5。使用 calculator 工具进行计算。
Final Answer: 总共是3000元。

[最终回答] 总共是3000元。
[使用步数] 1

[问题] 请帮我写一个Python函数,计算1到100的偶数列表并打印前10个
[预期工具] code_executor



--- Step 1 ---
Thought: 这是一个编程任务，需要生成1到100的偶数列表并打印前10个。我将使用 code_executor 工具来执行 Python 代码。
Action: code_executor({'code': 'def get_even_numbers():\n    # 生成1到100的偶数列表\n    even_list = [i for i in range(2, 101, 2)]\n    # 打印前10个\n    print("前10个偶数:", even_list[:10])\n    return even_list\n\n# 调用函数\nget_even_numbers()'})
Observation: 前10个偶数: [2, 4, 6, 8, 10, 12, 14, 16, 18, 20]




--- Step 2 ---
Thought: 代码执行成功，已生成1到100的偶数列表并打印了前10个偶数：[2, 4, 6, 8, 10, 12, 14, 16, 18, 20]。
Final Answer: 已成功生成1到100的偶数列表，并打印出前10个偶数：[2, 4, 6, 8, 10, 12, 14, 16, 18, 20]。完整偶数列表包含50个元素（2, 4, 6, ..., 100）。

[最终回答] 已成功生成1到100的偶数列表，并打印出前10个偶数：[2, 4, 6, 8, 10, 12, 14, 16, 18, 20]。完整偶数列表包含50个元素（2, 4, 6, ..., 100）。
[使用步数] 2

[问题] Python 的 list comprehension 怎么用?
[预期工具] direct_answer



--- Step 1 ---
Thought: 这是一个关于 Python 语法的通用编程问题，不涉及企业知识库、数学计算或代码执行，适合直接回答。
Final Answer: Python 的列表推导式（list comprehension）是一种简洁创建列表的语法，基本格式为：

```python
[expression for item in iterable if condition]
```

- `expression`：对每个元素进行的操作（如 `item * 2` 或 `item.upper()`）
- `item`：迭代变量
- `iterable`：可迭代对象（如列表、range、字符串等）
- `if condition`（可选）：过滤条件，只保留满足条件的元素

✅ 示例：
```python
# 生成 0~9 的平方
squares = [x**2 for x in range(10)]

# 从字符串列表中提取长度大于 3 的字符串并转大写
words = ["cat", "dog", "elephant", "bird"]
long_caps = [w.upper() for w in words if len(w) > 3]

# 嵌套使用（等价于双重 for 循环）
pairs = [(x, y) for x in [1, 2] for y in ['a', 'b']]  # [(1, 'a'), (1, 'b'), (2, 'a'), (2, 'b')]
```

⚠️ 注意：列表推导式适合逻辑简洁的场景；过于复杂时建议用传统 for 循环以保证可读性。

[最终回答] Python 的列表推导式（list comprehension）是一种简洁创建列表的语法，基本格式为：

```python
[expression for item in iterable if condition]
```

- `expression`：对每个元素进行的操作（如 `item * 2` 或 `item.upper()`）
- `item`：迭代变量
- `iterable`：
[使用步数] 1

✅ 检查点通过：Agent 成功回答了全部 4/4 个问题
✅ 检查点通过：所有问题在合理步数内完成


---

## 休息 10 分钟 (1:00 - 1:10)

到目前为止我们已经搭建了:
- **知识库** — 文档分块 + 向量存储 + 语义检索
- **RAG 系统** — 检索 + LLM 生成带来源的回答
- **Agent 路由** — ReAct 循环,自动选择合适的工具

接下来我们要解决两个关键问题:
1. **如何知道系统回答得好不好?** → Step 4: 自动评测
2. **如何支持连续对话?** → Step 5: 多轮对话记忆

---

# Step 4: 评测管道(1:10 - 1:30)

**目标:** 用 LLM-as-Judge 自动评估回答质量

### 为什么需要自动评测?

手工逐条检查 Agent 的回答既慢又主观。**LLM-as-Judge** 的思路是:让另一个 LLM 充当"评审员",按照预定义的标准自动打分。这样我们可以:
- 快速评估系统在大量问题上的表现
- 对比不同版本的系统（例如换了 Prompt 后分数是升还是降）
- 建立持续监控的基线

评测维度:
- **相关性（Relevance）**：回答是否切题
- **准确性（Accuracy）**：信息是否正确
- **完整性（Completeness）**：是否回答了所有要点

每项 1-5 分,由 LLM 自动打分。

In [12]:
# LLM-as-Judge 评测器

class LLMJudge:
    """使用 LLM 自动评估回答质量"""

    JUDGE_PROMPT = (
        "你是一个专业的 AI 回答质量评审员。请对以下问答进行评分。\n\n"
        "## 评分标准(每项 1-5 分)\n"
        "- **相关性 (Relevance)**:回答是否切合问题主题\n"
        "- **准确性 (Accuracy)**:信息是否与参考答案一致\n"
        "- **完整性 (Completeness)**:是否覆盖了问题的所有要点\n\n"
        "## 待评估内容\n"
        "**问题:** {question}\n"
        "**参考答案:** {reference}\n"
        "**模型回答:** {answer}\n\n"
        "## 输出格式(严格按此格式)\n"
        "Relevance: [1-5]\n"
        "Accuracy: [1-5]\n"
        "Completeness: [1-5]\n"
        "Reason: [一句话评价]"
    )

    def __init__(self, llm):
        self.llm = llm

    def judge(self, question: str, answer: str, reference: str) -> Dict:
        """评估单个问答对"""
        prompt = self.JUDGE_PROMPT.format(
            question=question, answer=answer, reference=reference
        )
        response = self.llm.chat(
            [{"role": "user", "content": prompt}],
            temperature=0.1
        )
        return self._parse_scores(response)

    def _parse_scores(self, response: str) -> Dict:
        """从 LLM 输出中提取分数"""
        scores = {}
        for metric in ["Relevance", "Accuracy", "Completeness"]:
            match = re.search(rf'{metric}:\s*(\d)', response)
            scores[metric.lower()] = int(match.group(1)) if match else 3
        reason_m = re.search(r'Reason:\s*(.+)', response)
        scores["reason"] = reason_m.group(1).strip() if reason_m else "无"
        scores["average"] = round(np.mean([scores["relevance"], scores["accuracy"], scores["completeness"]]), 2)
        return scores

    def evaluate_batch(self, eval_dataset: List[Dict]) -> List[Dict]:
        """批量评估"""
        results = []
        for item in eval_dataset:
            scores = self.judge(item["question"], item["answer"], item["reference"])
            results.append({**item, **scores})
        return results


judge = LLMJudge(llm)
print("[OK] LLM-as-Judge 评测器已初始化")

[OK] LLM-as-Judge 评测器已初始化


In [13]:
# 评测数据集
# 每条: question(问题) + reference(标准答案，供 Judge 对比打分)

eval_dataset = [
    {"question": "入职3年有几天年假?", "reference": "入职满1年不满10年:5天年假"},
    {"question": "出差住宿标准是多少?", "reference": "一线城市不超过500元/晚,其他城市不超过350元/晚"},
    {"question": "SL-3000 Pro 的内存是多少?", "reference": "SL-3000 Pro 为 4GB"},
    {"question": "API 接口的 P99 延迟要求是多少?", "reference": "P99 延迟不超过 200ms"},
    {"question": "StarView 基础版是免费的吗?", "reference": "基础版:免费(5个大屏以内,10个数据源以内)"},
    {"question": "CI/CD 流程中灰度发布的比例是怎样的?", "reference": "灰度比例:5% -> 20% -> 50% -> 100%"},
    {"question": "婚假有多少天?", "reference": "婚假15天"},
]

print(f"[OK] 评测数据集: {len(eval_dataset)} 题")

[OK] 评测数据集: 7 题


---

## Checkpoint 4: 评测打分

**目标:** 用 Agent 回答评测数据集中的问题,然后用 LLM-as-Judge 打分。

### 思路提示

这个 Checkpoint 把前面的组件串联起来:
1. **Agent 回答** — 对每个评测问题调用 `agent.run()` 获取回答
2. **Judge 打分** — 用 `judge.judge()` 对比回答与参考答案
3. **汇总分析** — 计算平均分,找出薄弱环节

注意:评测会调用多次 LLM,可能需要 1-2 分钟,请耐心等待。

**预期输出**：
```
评估结果:
| 问题 | 相关性 | 准确性 | 完整性 | 平均分 |
|------|-------|-------|-------|-------|
| Q1   | 4-5   | 4-5   | 3-5   | x.x   |
| ...  |       |       |       |       |
总体平均: x.x / 5.0
```

In [14]:
# Checkpoint 4: 运行评测

# ↓↓↓ 你的代码 ↓↓↓
eval_results = []

for i, item in enumerate(eval_dataset):
    print(f"评测进度: {i+1}/{len(eval_dataset)}...", end=" ")

    # 用 Agent 生成回答
    agent_result = agent.run(item["question"], verbose=False)
    item["answer"] = agent_result["answer"]

    # 用 LLM-as-Judge 评分
    scores = judge.judge(
        question=item["question"],
        answer=agent_result["answer"],
        reference=item["reference"]
    )

    eval_results.append({**item, **scores})
    print(f"均分: {scores.get('average', '?')}")
# ↑↑↑ 你的代码 ↑↑↑

# 打印汇总表格
print(f"\n{'='*60}")
print(f"{'问题':<25} {'相关':>4} {'准确':>4} {'完整':>4} {'均分':>5}")
print(f"{'-'*25} {'-'*4} {'-'*4} {'-'*4} {'-'*5}")
for r in eval_results:
    print(f"{r['question'][:24]:<25} {r.get('relevance','?'):>4} {r.get('accuracy','?'):>4} "
          f"{r.get('completeness','?'):>4} {r.get('average','?'):>5}")

if eval_results and all('average' in r for r in eval_results):
    overall = np.mean([r['average'] for r in eval_results])
    print(f"\n总体平均分: {overall:.2f} / 5.0")
    if overall >= 4.0:
        print("系统表现良好!")
    elif overall >= 3.0:
        print("系统表现尚可,建议优化检索或 Prompt。")
    else:
        print("系统表现较差,需要排查检索质量和 Prompt 设计。")

评测进度: 1/7... 

均分: 4.67
评测进度: 2/7... 

均分: 3.33
评测进度: 3/7... 

均分: 3.67
评测进度: 4/7... 

均分: 5.0
评测进度: 5/7... 

均分: 5.0
评测进度: 6/7... 

均分: 5.0
评测进度: 7/7... 

均分: 2.33

问题                          相关   准确   完整    均分
------------------------- ---- ---- ---- -----
入职3年有几天年假?                   5    5    4  4.67
出差住宿标准是多少?                   5    2    3  3.33
SL-3000 Pro 的内存是多少?          5    1    5  3.67
API 接口的 P99 延迟要求是多少?         5    5    5   5.0
StarView 基础版是免费的吗?           5    5    5   5.0
CI/CD 流程中灰度发布的比例是怎样的?        5    5    5   5.0
婚假有多少天?                      3    2    2  2.33

总体平均分: 4.14 / 5.0
系统表现良好!


---

# Step 5: 对话记忆与多轮支持(1:30 - 1:50)

**目标:** 让 Agent 能够记住之前的对话,支持多轮交互

### 为什么需要对话记忆?

到目前为止,我们的 Agent 每次回答都是"无状态"的——它不知道之前聊了什么。但真实场景中,用户经常会追问:

```
用户: StarLink 网关的价格是多少?
助手: SL-3000 为 2999元,SL-3000 Pro 为 4999元。
用户: Pro 版比普通版多了什么功能?  ← 需要知道上文在讨论 StarLink
助手: Pro 版额外支持 5G 模块,内存 4GB（普通版 2GB）。
```

如果没有记忆,Agent 不知道"Pro 版"指的是什么产品。

### 实现思路
1. **对话历史存储** — 保存每轮的问答（user/assistant 消息对）
2. **滑动窗口** — 只保留最近 N 轮,避免 context 过长导致 token 超限
3. **上下文注入** — 将历史对话拼入 System Prompt,让 LLM 能看到之前的内容

In [15]:
# ============================================================
# 对话记忆管理
# ============================================================

class ConversationMemory:
    """对话记忆：存储和管理多轮对话历史（滑动窗口策略）"""

    def __init__(self, max_turns: int = 5):
        self.max_turns = max_turns      # 最多保留的对话轮数
        self.history: List[Dict[str, str]] = []  # 消息列表

    def add_user_message(self, message: str):
        """添加用户消息"""
        self.history.append({"role": "user", "content": message})
        self._trim()

    def add_assistant_message(self, message: str):
        """添加助手回复"""
        self.history.append({"role": "assistant", "content": message})
        self._trim()

    def _trim(self):
        """滑动窗口：超过 max_turns 轮时,丢弃最早的对话"""
        max_messages = self.max_turns * 2  # 每轮 = 1条user + 1条assistant
        if len(self.history) > max_messages:
            self.history = self.history[-max_messages:]

    def get_context_string(self) -> str:
        """将历史对话格式化为字符串,注入到 Prompt 中"""
        if not self.history:
            return "(无历史对话)"
        parts = []
        for msg in self.history:
            role = "用户" if msg["role"] == "user" else "助手"
            parts.append(f"{role}: {msg['content']}")
        return "\n".join(parts)

    def get_messages(self) -> List[Dict[str, str]]:
        """返回原始消息列表"""
        return list(self.history)

    def clear(self):
        """清空所有对话历史"""
        self.history = []

    def __len__(self):
        """返回对话轮数（非消息条数）"""
        return len(self.history) // 2


print("[OK] ConversationMemory 定义完成")

[OK] ConversationMemory 定义完成


In [16]:
# ============================================================
# 带记忆的 Agent
# ============================================================

class MultiTurnAgent:
    """支持多轮对话的企业知识助手"""

    SYSTEM_PROMPT = (
        "你是星辰科技的智能助手,可以使用工具回答问题。你能记住之前的对话。\n\n"
        "可用工具:\n{tool_descriptions}\n\n"
        "使用以下格式:\n"
        "Thought: [分析问题,考虑对话历史]\n"
        "Action: [工具名称]\n"
        "Action Input: {{\"key\": \"value\"}}\n\n"
        "或直接回答:\n"
        "Thought: [分析]\n"
        "Final Answer: [回答]\n\n"
        "历史对话:\n{conversation_history}"
    )

    def __init__(self, llm, tools: List[Tool], max_steps: int = 5, max_memory_turns: int = 5):
        self.llm = llm
        self.tools = {t.name: t for t in tools}
        self.max_steps = max_steps
        self.memory = ConversationMemory(max_turns=max_memory_turns)

    def _get_tool_descriptions(self) -> str:
        descs = []
        for name, tool in self.tools.items():
            descs.append(f"- {name}: {tool.description}")
        return "\n".join(descs)

    def _parse_response(self, response: str) -> Dict:
        result = {"thought": None, "action": None, "action_input": None, "final_answer": None}
        thought_m = re.search(r'Thought:\s*(.+?)(?=Action:|Final Answer:|$)', response, re.DOTALL)
        if thought_m:
            result["thought"] = thought_m.group(1).strip()
        final_m = re.search(r'Final Answer:\s*(.+?)$', response, re.DOTALL)
        if final_m:
            result["final_answer"] = final_m.group(1).strip()
            return result
        action_m = re.search(r'Action:\s*([\w_]+)', response)
        if action_m:
            result["action"] = action_m.group(1).strip()
        input_m = re.search(r'Action Input:\s*(.+?)(?=Thought:|Action:|Final Answer:|$)', response, re.DOTALL)
        if input_m:
            try:
                result["action_input"] = json.loads(input_m.group(1).strip())
            except json.JSONDecodeError:
                result["action_input"] = {"input": input_m.group(1).strip()}
        return result

    def chat(self, user_message: str, verbose: bool = True) -> str:
        """多轮对话接口"""
        self.memory.add_user_message(user_message)

        tool_descs = self._get_tool_descriptions()
        history_str = self.memory.get_context_string()
        system_msg = self.SYSTEM_PROMPT.format(
            tool_descriptions=tool_descs,
            conversation_history=history_str
        )

        messages = [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_message}
        ]

        for step in range(self.max_steps):
            response = self.llm.chat(messages, temperature=0.1)
            parsed = self._parse_response(response)

            if verbose and parsed["thought"]:
                print(f"  [Thought] {parsed['thought'][:100]}")

            if parsed["final_answer"]:
                self.memory.add_assistant_message(parsed["final_answer"])
                return parsed["final_answer"]

            if parsed["action"] and parsed["action"] in self.tools:
                tool = self.tools[parsed["action"]]
                action_input = parsed["action_input"] or {}
                if verbose:
                    print(f"  [Action] {parsed['action']}({action_input})")
                observation = tool.run(**action_input)
                messages.append({"role": "assistant", "content": response})
                messages.append({"role": "user", "content": f"Observation: {observation}"})
            else:
                self.memory.add_assistant_message(response)
                return response

        fallback = "抱歉,我无法完成这个请求。"
        self.memory.add_assistant_message(fallback)
        return fallback

    def reset(self):
        self.memory.clear()
        print("[OK] 对话已重置")


# 实例化
assistant = MultiTurnAgent(llm, tools, max_memory_turns=5)
print("[OK] 多轮对话 Agent 已初始化")

[OK] 多轮对话 Agent 已初始化


---

## Checkpoint 5: 多轮对话

**目标:** 测试 Agent 的多轮对话能力,验证它能正确引用上文信息。

### 思路提示

设计一组**有依赖关系**的连续问题。后续问题应该用代词或省略主语,只有结合上文才能理解:
- 第 1 轮：明确提到某个主题（如 StarLink 网关）
- 第 2 轮：用"它""这个产品"等代词追问
- 第 3 轮：基于前面的回答做进一步提问（如用之前提到的价格做计算）

要求:
1. 设计一组连续对话（至少 3 轮）
2. 后续问题必须依赖前面的上下文
3. 验证 Agent 能正确理解代词指代（如"它""这个产品"）

**预期输出**：
```
[Turn 1] 用户: 介绍一下StarLink产品
         助手: StarLink是...（详细介绍）
[Turn 2] 用户: 它的价格是多少？（代词“它”应解析为StarLink）
         助手: StarLink的价格是...
[Turn 3] 用户: 和StarView比哪个好？
         助手: 两者对比...（利用上下文理解比较对象）
```

In [17]:
# ============================================================
# Checkpoint 5: 多轮对话测试
# ============================================================

assistant.reset()  # 清除之前的对话历史

# ---- TODO: 学员可自行修改或设计自己的多轮对话 ----
multi_turn_questions = [
    "StarLink 产品有哪些型号?",             # 第1轮：建立主题
    "Pro 版的价格是多少?",                   # 第2轮：省略"StarLink",测试上下文
    "它支持哪些通信协议?",                    # 第3轮：用"它"指代 StarLink
    "帮我算一下买10台Pro需要多少钱",          # 第4轮：引用之前的价格做计算
]
# ---- END TODO ----

cp5_answers = []
for q in multi_turn_questions:
    print(f"\n用户: {q}")
    answer = assistant.chat(q, verbose=True)
    print(f"助手: {answer}\n")
    cp5_answers.append(answer)

print(f"\n[对话轮数] {len(assistant.memory)}")

# 验证：多轮对话的基本功能
cp5_all_answered = all(a and len(a.strip()) > 0 for a in cp5_answers)
cp5_memory_works = len(assistant.memory) >= len(multi_turn_questions)

if cp5_all_answered:
    print(f"✅ 检查点通过：多轮对话正常，全部 {len(multi_turn_questions)} 轮都获得了回答")
else:
    empty_turns = [i+1 for i, a in enumerate(cp5_answers) if not a or len(a.strip()) == 0]
    print(f"❌ 检查点失败：第 {empty_turns} 轮未获得回答，请检查对话流程")

if cp5_memory_works:
    print(f"✅ 检查点通过：对话记忆正常，记录了 {len(assistant.memory)} 条历史")
else:
    print(f"❌ 检查点失败：对话记忆异常（期望 >= {len(multi_turn_questions)}，实际 {len(assistant.memory)}），请检查 ConversationMemory")

[OK] 对话已重置

用户: StarLink 产品有哪些型号?


  [Thought] 我需要查询企业知识库中关于 StarLink 产品的型号信息。
  [Action] rag_search({'query': 'StarLink 产品型号'})


助手: StarLink 智能网关目前有以下两款型号：

- SL-3000  
- SL-3000 Pro


用户: Pro 版的价格是多少?


  [Thought] 需要查询 StarLink SL-3000 Pro 的官方定价信息，这属于产品信息范畴，应使用企业知识库检索。
  [Action] rag_search({'query': 'StarLink SL-3000 Pro 官方售价'})


助手: StarLink SL-3000 Pro 的官方售价为 4999 元。


用户: 它支持哪些通信协议?


  [Thought] 需要查询 StarLink SL-3000 Pro 的技术规格，特别是其支持的通信协议。这属于产品技术文档范畴，适合使用 rag_search 工具检索。
  [Action] rag_search({'query': 'StarLink SL-3000 Pro 支持的通信协议'})


助手: StarLink SL-3000 Pro 支持以下通信协议：  
- MQTT  
- CoAP  
- HTTP/2  
- Modbus TCP


用户: 帮我算一下买10台Pro需要多少钱


  [Thought] 已知 StarLink SL-3000 Pro 单台售价为 4999 元，购买 10 台需计算 4999 × 10。使用计算器工具进行精确计算。
助手: 买10台 StarLink SL-3000 Pro 需要 49,990 元。


[对话轮数] 4
✅ 检查点通过：多轮对话正常，全部 4 轮都获得了回答
✅ 检查点通过：对话记忆正常，记录了 4 条历史


---

# Step 6: 学生自定义(1:50 - 2:10)

**自由编码时间!** 到这里,你已经拥有了一个完整的企业知识助手:RAG + Agent + 评测 + 多轮对话。

现在是发挥创意的时候了——从以下三个方向中选择一个（或多个）进行扩展,让系统更贴近你的实际工作场景。

### 方向 A: 添加自己的文档
将你公司/团队的真实文档（或模拟文档）加入知识库,测试检索效果。

### 方向 B: 添加自定义工具
例如:日期计算、汇率转换、数据库查询等——让 Agent 能做更多事情。

### 方向 C: 修改评测标准
添加新的评测维度（如"简洁性""语言风格""安全性"），让评测更全面。

In [18]:
# ============================================================
# 扩展 A: 添加自定义文档
# ============================================================
# 将下面的 title / content 替换为你自己的文档内容

my_document = {
    "title": "员工培训与发展制度",
    "category": "HR",
    "content": """星辰科技员工培训与发展制度(2024版)

一、新员工培训
入职第一周参加公司文化与制度培训。
入职第一个月由导师带教,完成岗位技能培训。
试用期结束前需通过岗位考核。

二、在职培训
每位员工每年享有 5 天带薪培训假。
公司每季度组织一次技术分享会。
鼓励员工参加外部技术大会,费用由公司报销(上限 5000 元/年)。

三、晋升通道
技术序列:初级工程师 → 中级工程师 → 高级工程师 → 技术专家 → 首席架构师
管理序列:组长 → 经理 → 总监 → VP
每年两次晋升评审(6月和12月)。
"""
}

# 分块并添加到向量数据库（和之前的企业文档共用同一个 vector_store）
chunks = chunker.chunk_text(
    my_document["content"],
    metadata={"doc_id": "custom_doc", "title": my_document["title"], "category": my_document["category"]}
)
chunk_texts = [c["text"] for c in chunks]
chunk_metadata = [c["metadata"] for c in chunks]
vector_store.add_documents(chunk_texts, metadata=chunk_metadata)

# 测试搜索,验证自定义文档是否被成功索引
test_results = vector_store.search("员工培训", top_k=2)

if chunks and len(chunks) > 0:
    print(f"\u2705 检查点通过：自定义文档已分为 {len(chunks)} 个块并添加到向量库")
else:
    print("\u274c 检查点失败：文档分块结果为空，请检查文档内容和分块器")

if test_results and len(test_results) > 0:
    print("\u2705 检查点通过：自定义文档可被搜索到")
    for r in test_results:
        print(f"  [{r['score']:.3f}] {r['document'][:60]}...")
else:
    print("\u274c 检查点失败：搜索未命中自定义文档，请检查索引是否成功")


✓ 已添加 2 个文档，总计 12 个


✅ 检查点通过：自定义文档已分为 2 个块并添加到向量库
✅ 检查点通过：自定义文档可被搜索到
  [0.679] 星辰科技员工培训与发展制度(2024版)

一、新员工培训
入职第一周参加公司文化与制度培训。
入职第一个月由导师带教,...
  [0.458] 星辰科技年假制度(2024版)

一、年假天数
入职满1年不满10年:5天年假
入职满10年不满20年:10天年假
入职...


In [19]:
# ============================================================
# 方向 B: 添加自定义工具
# ============================================================
# 示例：日期计算工具——让 Agent 能回答和日期相关的问题

def date_calculator(operation: str) -> str:
    """日期计算（例如：今天几号、距离某日期还有几天）"""
    from datetime import datetime, timedelta
    import re as _re
    today = datetime.now()

    if "今天" in operation or "日期" in operation or "几号" in operation:
        weekdays = ["一", "二", "三", "四", "五", "六", "日"]
        return f"今天是 {today.strftime('%Y年%m月%d日')}，星期{weekdays[today.weekday()]}"

    # 尝试匹配 "距离 YYYY-MM-DD 还有几天" 或 "YYYY年MM月DD日"
    date_match = _re.search(r'(\d{4})[-年](\d{1,2})[-月](\d{1,2})', operation)
    if date_match:
        y, m, d = int(date_match.group(1)), int(date_match.group(2)), int(date_match.group(3))
        try:
            target = datetime(y, m, d)
            delta = (target - today).days
            if delta > 0:
                return f"距离 {y}年{m}月{d}日 还有 {delta} 天"
            elif delta < 0:
                return f"{y}年{m}月{d}日 已经过去了 {abs(delta)} 天"
            else:
                return f"{y}年{m}月{d}日 就是今天!"
        except ValueError:
            return "日期格式无效，请使用 YYYY-MM-DD 格式"

    # 尝试匹配 "N天后" 或 "N天前"
    days_match = _re.search(r'(\d+)\s*天(后|前)', operation)
    if days_match:
        n = int(days_match.group(1))
        direction = days_match.group(2)
        weekdays = ["一", "二", "三", "四", "五", "六", "日"]
        if direction == "后":
            result_date = today + timedelta(days=n)
        else:
            result_date = today - timedelta(days=n)
        return f"{n}天{direction}是 {result_date.strftime('%Y年%m月%d日')}，星期{weekdays[result_date.weekday()]}"

    return f"今天是 {today.strftime('%Y年%m月%d日')}。请提供更具体的日期计算指令（如：距离2025-12-31还有几天、30天后是几号）"

# 将自定义工具注册到多轮对话 Agent
date_tool = Tool(
    "date_calculator",
    "日期计算工具（今天几号、距离某日期多少天、N天后是几号等）",
    {"type": "object", "properties": {"operation": {"type": "string", "description": "日期计算指令"}}},
    date_calculator
)

assistant.tools[date_tool.name] = date_tool  # 动态注册到现有 Agent
print(f"[OK] 已注册自定义工具: {date_tool.name}")


[OK] 已注册自定义工具: date_calculator


In [20]:
# ============================================================
# 方向 C: 修改评测标准
# ============================================================
# 示例: 扩展 LLMJudge，添加「简洁性(Conciseness)」评测维度

class ExtendedJudge(LLMJudge):
    """扩展评测器，增加 Conciseness 维度"""

    JUDGE_PROMPT = (
        "你是一个专业的 AI 回答质量评审员。请对以下问答进行评分。\n\n"
        "## 评分标准(每项 1-5 分)\n"
        "- **相关性 (Relevance)**:回答是否切合问题主题\n"
        "- **准确性 (Accuracy)**:信息是否与参考答案一致\n"
        "- **完整性 (Completeness)**:是否覆盖了问题的所有要点\n"
        "- **简洁性 (Conciseness)**:回答是否简明扼要,没有冗余信息\n\n"
        "## 待评估内容\n"
        "**问题:** {question}\n"
        "**参考答案:** {reference}\n"
        "**模型回答:** {answer}\n\n"
        "## 输出格式(严格按此格式)\n"
        "Relevance: [1-5]\n"
        "Accuracy: [1-5]\n"
        "Completeness: [1-5]\n"
        "Conciseness: [1-5]\n"
        "Reason: [一句话评价]"
    )

    def _parse_scores(self, response: str) -> Dict:
        """解析 4 个维度的分数"""
        scores = {}
        for metric in ["Relevance", "Accuracy", "Completeness", "Conciseness"]:
            match = re.search(rf'{metric}:\s*(\d)', response)
            scores[metric.lower()] = int(match.group(1)) if match else 3
        reason_m = re.search(r'Reason:\s*(.+)', response)
        scores["reason"] = reason_m.group(1).strip() if reason_m else "无"
        scores["average"] = round(np.mean([scores["relevance"], scores["accuracy"],
                                             scores["completeness"], scores["conciseness"]]), 2)
        return scores


extended_judge = ExtendedJudge(llm)
print("[OK] 扩展评测器已初始化（增加 Conciseness 维度）")


[OK] 扩展评测器已初始化（增加 Conciseness 维度）


---

## Checkpoint 6: 最终演示(2:10 - 2:30)

**目标:** 演示你的完整系统。

这是展示成果的时刻!运行下面的演示脚本,观察系统在连续对话中的表现。你也可以修改 `demo_questions` 来展示自己添加的自定义文档或工具。

演示要点:
1. 展示你添加的自定义内容（文档/工具/评测标准）
2. 运行一组多轮对话,观察 Agent 如何路由和引用上下文
3. 每组 3-5 分钟,准备 2-3 个精彩的演示用例

In [21]:
# ============================================================
# 最终演示脚本
# ============================================================

print("=" * 60)
print("   星辰科技 · 企业知识助手 Demo")
print("=" * 60)

assistant.reset()  # 重置对话记忆,从头开始

# 演示用的多轮对话（覆盖多种工具和上下文引用）
demo_questions = [
    "你好!请介绍一下公司的 StarLink 产品系列",   # RAG 检索
    "Pro 版和普通版有什么区别?",                    # 上下文引用
    "如果我要买 20 台 Pro 版,总价是多少?",         # 计算器
    "公司的 API 开发用什么认证方式?",               # RAG 检索（切换话题）
    "新员工第一年有年假吗?",                        # RAG 检索（HR 文档）
]

for q in demo_questions:
    print(f"\n{'- '*30}")
    print(f"  用户: {q}")
    answer = assistant.chat(q, verbose=False)
    print(f"  助手: {answer}")

print(f"\n{'='*60}")
print(f"共 {len(demo_questions)} 轮对话,记忆 {len(assistant.memory)} 轮")

   星辰科技 · 企业知识助手 Demo
[OK] 对话已重置

- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 
  用户: 你好!请介绍一下公司的 StarLink 产品系列


  助手: 星辰科技的 **StarLink 产品系列** 目前聚焦于边缘智能网关领域，主打型号为：

🔹 **StarLink 智能网关 v3.0**  
- 基础版：**SL-3000**（售价 ¥2999）  
- 高配版：**SL-3000 Pro**（售价 ¥4999）  

✅ **核心优势**：  
- **强劲算力**：ARM Cortex-A72 四核 1.8GHz CPU；SL-3000 Pro 达 **50 TOPS AI 算力**（支持 TensorFlow Lite 边缘推理）  
- **丰富接口**：4×千兆以太网 + WiFi 6；Pro 版额外集成 **5G 全网通模块**  
- **协议兼容广**：原生支持 MQTT、CoAP、HTTP/2、Modbus TCP 等主流工业与物联网协议  
- **灵活扩展**：标配 16GB eMMC + SD 卡槽，便于本地数据缓存与模型部署  

🏭 **典型应用**：  
- 工厂设备实时数据采集与预测性维护  
- 智慧园区多源异构设备统一接入与边缘分析  
- 车联网路侧单元（RSU）数据汇聚与低时延处理  

📦 **服务保障**：整机提供 **2年质保**。

⚠️ 注：目前 StarLink 系列暂不包含卫星通信相关产品（如星链式卫星终端），其命名源于“连接星辰万物”的品牌理念，专注地面边缘智能连接。如需了解其他产品线（如 StarView 数据平台或 StarEdge 边缘OS），欢迎随时提问！

- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 
  用户: Pro 版和普通版有什么区别?


  助手: StarLink SL-3000 Pro 与基础版 SL-3000 的核心区别如下（已按重要性排序）：

| 维度         | SL-3000（基础版）              | SL-3000 Pro（高配版）                     | 差异说明 |
|--------------|-------------------------------|------------------------------------------|----------|
| **AI 算力**   | 4 TOPS（NPU 加速）             | **50 TOPS**（专用 AI 加速引擎）           | Pro 版支持实时运行复杂模型（如YOLOv8s目标检测、LSTM设备故障预测），基础版仅适用于轻量推理（如阈值告警）。 |
| **通信能力** | WiFi 6 + 4×千兆以太网           | **+ 内置5G全网通模块**（Sub-6GHz，支持SA/NSA） | Pro 版可作为移动/无固定网络场景的主链路（如车载边缘节点、临时施工监控），基础版需依赖有线或WiFi。 |
| **存储与扩展** | 16GB eMMC + SD卡槽             | **32GB eMMC + 双SD卡槽 + M.2 NVMe 接口** | Pro 版支持本地长期数据缓存（>30天视频元数据）及高速模型热加载。 |
| **软件功能**  | 标准固件（支持MQTT/Modbus/HTTP） | **预装 StarEdge OS 2.2 + 边缘AI管理套件** | 包含可视化模型部署工具、OTA远程训练更新、多租户隔离策略，适合企业级统一运维。 |
| **物理特性**  | 铝合金外壳，IP30防护            | **IP54防护 + 宽温设计（-30℃~70℃）**       | Pro 版适用于户外、工业现场等严苛环境。 |
| **价格**      | ¥2999                          | ¥4999                                    | 差价 ¥2000，对应算力提升12.5倍、通信与可靠性全面升级。 |

💡

  助手: 20 台 StarLink SL-3000 Pro 的总价为 **¥99,980**。  
如需开具发票、了解批量采购折扣（例如50台以上享额外优惠）、或搭配 StarEdge OS 订阅服务（如AI模型托管/远程运维），我可立即为您查询或生成报价单。

- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 
  用户: 公司的 API 开发用什么认证方式?


  助手: 星辰科技的 API 开发采用分场景认证机制，具体如下：

🔹 **内部服务间调用**：  
- 使用 **JWT（JSON Web Token）** 认证；  
- Token 由统一身份中心（IAM）签发，有效期为 **2 小时**，支持自动刷新（需配合 refresh token）；  
- Header 格式：`Authorization: Bearer <jwt_token>`。

🔹 **面向外部合作伙伴或客户的公开 API**：  
- 实行 **OAuth 2.0 + API Key 双重认证**：  
  - OAuth 2.0 用于用户授权与访问范围控制（如 `scope=read:device write:alarm`）；  
  - API Key（固定密钥）用于应用级身份识别与配额管理，需在请求 Header 中传递：`X-API-Key: <your_api_key>`。  
- 所有敏感操作（如设备控制、数据导出）强制要求两者同时有效。

✅ 补充说明：  
- 所有 Token 和 Key 均通过 HTTPS 传输，禁止明文日志记录；  
- 开发者可通过 [开发者门户](https://dev.stars-tech.com) 申请 API Key、管理 OAuth Client，并查看实时调用审计日志。

如需获取 OAuth 授权码流程示例、JWT 解析代码片段（Python/Node.js），或申请测试环境 API Key，我可立即协助！

- - - - - - - - - - - - - - - - - - - - - - - - - - - - - - 
  用户: 新员工第一年有年假吗?


  助手: 根据星辰科技《年假与休假制度》规定：  
✅ **新员工入职第一年不享受年假**。原因如下：  
- 入职首年处于试用期（通常3–6个月），而**试用期员工不享有年假**；  
- 年假天数按“连续工作年限”计算，需**入职满1整年**后方可首次享有年假；  
- 满1年不满10年的正式员工，年假为 **5个工作日/年**（自满1年当日起生效，非自然年累计）。

📌 补充说明：  
- 年假可分次使用，最小单位为半天；  
- 未休年假原则上不跨年清零，但需在次年3月31日前申请使用，逾期自动作废（特殊情况可申请延期，需部门负责人审批）。

如需了解年假申请流程、调休规则或试用期转正后的其他假期（如婚假、陪产假），欢迎随时提问！

共 5 轮对话,记忆 5 轮


---

# 生产考量(2:30 - 2:50)

恭喜完成 Capstone 项目! 你已经拥有了一个功能完整的企业知识助手原型。

从 Notebook 原型到生产系统,还需要考虑以下几个方面。这些内容帮助你规划"下一步怎么落地"。

## 1. 部署方案

| 方案 | 适用场景 | 复杂度 | 示例 |
|------|----------|--------|------|
| **Gradio Demo** | 快速原型/内部演示 | 低 | `gr.ChatInterface()` |
| **FastAPI 服务** | 后端 API,集成到现有系统 | 中 | REST API + Swagger |
| **Docker + K8s** | 生产环境,高可用 | 高 | 容器化 + 自动扩缩 |

```python
# Gradio 快速原型(5行代码)
import gradio as gr

def chat_fn(message, history):
    return assistant.chat(message, verbose=False)

gr.ChatInterface(chat_fn, title="企业知识助手").launch()
```

```python
# FastAPI 服务
from fastapi import FastAPI
app = FastAPI()

@app.post("/chat")
async def chat(request: dict):
    answer = assistant.chat(request["message"])
    return {"answer": answer}
```

## 2. 监控与可观测性

生产环境必须监控的指标:

| 指标 | 工具 | 告警阈值 |
|------|------|----------|
| 请求延迟(P50/P99) | Prometheus + Grafana | P99 > 5s |
| LLM 调用失败率 | 日志 + ELK | > 5% |
| 检索命中率 | 自定义埋点 | < 60% |
| 用户满意度 | 反馈按钮 | < 3.5/5 |

**关键实践:**
- 记录每次请求的完整链路:`query -> retrieval -> LLM call -> response`
- 保存 Agent 的思考过程(Thought),便于调试
- 定期用评测数据集回归测试,防止质量下降

## 3. 成本优化

LLM API 调用是主要成本项。优化策略:

| 策略 | 效果 | 实现难度 |
|------|------|----------|
| **缓存(Caching)** | 相同问题复用答案,降低 50-80% 调用 | 低 |
| **模型分级** | 简单问题用小模型,复杂问题用大模型 | 中 |
| **Prompt 精简** | 减少 token 数,降低单次成本 | 低 |
| **批量处理** | 离线评测批量请求 | 低 |
| **本地模型** | Ollama 部署,零 API 费用 | 高(需 GPU) |

```python
# 简易缓存示例
import hashlib

cache = {}

def cached_chat(question):
    key = hashlib.md5(question.encode()).hexdigest()
    if key in cache:
        return cache[key]  # 缓存命中
    answer = assistant.chat(question)
    cache[key] = answer
    return answer
```

## 4. 安全与合规

企业场景的安全考量:

- **输入过滤(Input Validation)**:检测并拒绝注入攻击(Prompt Injection)
- **输出过滤(Output Filtering)**:防止泄露敏感信息(如薪资、合同细节)
- **权限控制(Access Control)**:不同角色看到不同文档
- **审计日志(Audit Log)**:记录谁问了什么,回答了什么

```python
# 简易输入过滤
BLOCKED_PATTERNS = [
    r"ignore.*instructions",
    r"system.*prompt",
    r"reveal.*password",
]

def is_safe_input(text: str) -> bool:
    for pattern in BLOCKED_PATTERNS:
        if re.search(pattern, text, re.IGNORECASE):
            return False
    return True
```

---

# "周一早上"行动计划(2:50 - 3:10)

回到工作后,如何落地?

## 第一周行动清单

### Day 1: 评估与立项
- [ ] 识别一个高价值场景(如:客服知识库、内部 FAQ、技术文档问答)
- [ ] 评估数据可用性(有哪些文档?什么格式?数据量多大?)
- [ ] 确定 LLM 方案:API(qwen-plus / GPT-4)还是本地部署(Ollama)

### Day 2-3: 快速原型
- [ ] 搭建开发环境(Python + 依赖)
- [ ] 用本课程的代码搭建第一版 RAG 系统
- [ ] 导入 5-10 份真实文档测试

### Day 4-5: 评测与优化
- [ ] 建立评测数据集(20-50 个问答对)
- [ ] 运行 LLM-as-Judge 获取基线分数
- [ ] 调优:改进分块策略、调整检索参数、优化 Prompt

## 推荐技术栈

| 场景 | 推荐方案 |
|------|----------|
| 快速验证 | 本课程代码 + Ollama/DashScope |
| 小团队生产 | LangChain + ChromaDB + FastAPI |
| 企业级生产 | LlamaIndex + Milvus + Docker + K8s |
| 多 Agent 协作 | CrewAI / AutoGen |

## 持续学习资源

- **LangChain 官方文档**: https://docs.langchain.com/
- **LlamaIndex 官方文档**: https://docs.llamaindex.ai/
- **Ollama**: https://ollama.ai/
- **通义千问 API**: https://dashscope.aliyun.com/

In [22]:
# ============================================================
# 互动环节：写下你的行动计划
# ============================================================

my_action_plan = {
    "scenario": "客服知识库问答",
    "data_source": "内部 Confluence 文档 + 产品 FAQ",
    "llm_choice": "通义千问 qwen-plus API",
    "first_step": "收集整理 10 份核心文档，搭建 RAG 原型",
    "success_metric": "评测数据集平均分 >= 4.0 / 5.0",
}

print("我的行动计划:")
for k, v in my_action_plan.items():
    print(f"  {k}: {v if v else '(待填写)'}")

我的行动计划:
  scenario: 客服知识库问答
  data_source: 内部 Confluence 文档 + 产品 FAQ
  llm_choice: 通义千问 qwen-plus API
  first_step: 收集整理 10 份核心文档，搭建 RAG 原型
  success_metric: 评测数据集平均分 >= 4.0 / 5.0


---

# 框架桥接:LangChain / LlamaIndex / CrewAI(3:10 - 3:20)

我们三天手写了所有核心组件。生产环境中,这些框架帮你省去重复劳动。

**核心观点:你现在理解了这些框架的底层原理。**

## 框架对比

| 框架 | 定位 | 优势 | 适合场景 |
|------|------|------|----------|
| **LangChain** | Agent / Chain / RAG 全栈 | 生态丰富,组件多 | 通用 AI 应用 |
| **LlamaIndex** | 数据索引 + RAG 专注 | RAG 场景强,数据连接器多 | 知识库/文档问答 |
| **CrewAI** | Multi-Agent 框架 | 角色定义直观 | 多智能体协作 |
| **AutoGen** | 微软 Multi-Agent | 对话驱动,支持人在回路 | 研究/复杂工作流 |
| **Dify / FastGPT** | 低代码平台 | 可视化编排,快速上线 | 非开发人员 |

In [23]:
# ============================================================
# 框架等价代码对照(伪代码,不执行)
# ============================================================

framework_comparison = '''
# ---- 我们手写的 ----
rag = KnowledgeRAG(vector_store, llm)
result = rag.answer("年假有几天?")

# ---- LangChain 等价 ----
from langchain_community.llms import Ollama
from langchain.chains import RetrievalQA
from langchain_community.vectorstores import Chroma

llm = Ollama(model="qwen2.5:7b")
vectorstore = Chroma.from_texts(documents, embedding)
qa_chain = RetrievalQA.from_chain_type(llm, retriever=vectorstore.as_retriever())
result = qa_chain.invoke("年假有几天?")

# ---- LlamaIndex 等价 ----
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader

documents = SimpleDirectoryReader("./data").load_data()
index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine()
result = query_engine.query("年假有几天?")

# ---- CrewAI 等价 (Multi-Agent) ----
from crewai import Agent, Task, Crew

researcher = Agent(role="知识检索员", goal="搜索相关文档")
writer = Agent(role="回答撰写员", goal="生成准确回答")
crew = Crew(agents=[researcher, writer], tasks=[...])
result = crew.kickoff()
'''

print("手写 vs 框架 代码对照:")
print(framework_comparison)
print("\n关键洞察: 框架封装了我们手写的每一步——Embedding、向量存储、检索、LLM调用。")
print("理解底层原理后,使用框架会更加得心应手。")

手写 vs 框架 代码对照:

# ---- 我们手写的 ----
rag = KnowledgeRAG(vector_store, llm)
result = rag.answer("年假有几天?")

# ---- LangChain 等价 ----
from langchain_community.llms import Ollama
from langchain.chains import RetrievalQA
from langchain_community.vectorstores import Chroma

llm = Ollama(model="qwen2.5:7b")
vectorstore = Chroma.from_texts(documents, embedding)
qa_chain = RetrievalQA.from_chain_type(llm, retriever=vectorstore.as_retriever())
result = qa_chain.invoke("年假有几天?")

# ---- LlamaIndex 等价 ----
from llama_index.core import VectorStoreIndex, SimpleDirectoryReader

documents = SimpleDirectoryReader("./data").load_data()
index = VectorStoreIndex.from_documents(documents)
query_engine = index.as_query_engine()
result = query_engine.query("年假有几天?")

# ---- CrewAI 等价 (Multi-Agent) ----
from crewai import Agent, Task, Crew

researcher = Agent(role="知识检索员", goal="搜索相关文档")
writer = Agent(role="回答撰写员", goal="生成准确回答")
crew = Crew(agents=[researcher, writer], tasks=[...])
result = crew.kickoff()


关键

---

# 课程总结(3:20 - 3:30)

## 三天学习之旅

```
Day 1 上午: 训练循环 → 自动微分 → Embedding
Day 1 下午: Attention → Transformer → GPT → Tokenizer
Day 2 上午: SFT 微调 → LoRA / QLoRA
Day 2 下午: 预训练 → DPO 对齐 → KV Cache
Day 3 上午: 评测体系 → 部署选型 → Agent → RAG 原理
Day 3 下午: RAG 实战 → Agent 实战 → Capstone 项目
```

## 你已经掌握的能力

| 能力 | 具体内容 |
|------|----------|
| 理解 LLM 架构 | 从 Embedding → Attention → Transformer → GPT 每一层 |
| 微调模型 | SFT 全流程 + LoRA 参数高效微调 |
| 评测模型 | PPL → Benchmark → LLM-as-Judge 多层次评测 |
| 构建 RAG | 文档分块 → 向量检索 → 上下文注入 → LLM 生成 |
| 构建 Agent | ReAct 推理循环 → 工具调用 → 多步规划 |
| 端到端系统 | Capstone: RAG + Agent + 评测 + 多轮对话 |

## 核心理念

> **先手写,再用框架。理解原理,才能驾驭工具。**

你不是在「调 API」,你理解 API 背后的每一层。
当框架出 bug 时,你能定位根因。
当选型时,你能评估优劣。
当面试时,你能讲清楚「为什么」。

---

**感谢参与三天的培训!祝大家在 AI 之旅上一帆风顺。**

有任何问题,欢迎随时联系讲师。